In [0]:
!pip install fredapi

In [0]:
jdbc_url = "jdbc:sqlserver://capstone-database-server.database.windows.net:1433;database=writedatabasesilverlayer;"
connection_properties = {
    "user": "capstonedioxieteam",
    "password": "Connhenbeo1@",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
from pyspark.sql.functions import col, last
from pyspark.sql.functions import expr
from pyspark.sql.types import DateType
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from datetime import timedelta
from datetime import datetime
from fredapi import Fred
import pandas as pd
import sys

class FredSparkLoader:
    def __init__(self, spark: SparkSession, api_key: str):
        self.spark = spark
        self.fred = Fred(api_key=api_key)

    def fetch_indicator(self, series_id: str, start: str = "2000-01-01", end: str = None) -> pd.DataFrame:
        if not end:
            end = datetime.today().strftime('%Y-%m-%d')
        data = self.fred.get_series(series_id, observation_start=start, observation_end=end)
        df = pd.DataFrame(data, columns=["value"])
        df["date"] = df.index
        df.reset_index(drop=True, inplace=True)
        return df[["date", "value"]]

    def to_spark_df(self, pandas_df: pd.DataFrame, indicator_name: str):
        df = self.spark.createDataFrame(pandas_df)
        return df.withColumnRenamed("value", f"{indicator_name}_value")

    def fetch_multiple_indicators(self, indicator_dict: dict, start: str = "2000-01-01", end: str = None) -> dict:
        spark_dfs = {}
        for name, series_id in indicator_dict.items():
            pd_df = self.fetch_indicator(series_id, start, end)
            spark_df = self.to_spark_df(pd_df, name)
            spark_dfs[name] = spark_df
        return spark_dfs


In [0]:
api_key = "2e130c627849486b2a30fb8e4bc0121d"
fred_loader = FredSparkLoader(spark, api_key)

macro_indicators = {
    "gdp": "GDP",
    "real_gdp": "GDPC1",
    "ferfed_funds_effective_rate": "FEDFUNDS",
    "labor_force_participant_rate": "CIVPART",
    "cpi": "CPIAUCSL",
    "unemployment": "UNRATE",
    "interest_rate": "FEDFUNDS",
    "job_openning_non_farm" : "JTSJOL",
    "hires_total_non_farm" : "JTSHIL",
    "quit_total_non_farm" : "JTSQUR",
    "layoff_discharge_non_farm" : "JTSLDL",
    "layoffs_and_discharges_professional_and_business_services" : "JTU540099LDL",
    "layoffs_and_discharges_manufacturing" : "JTU3000LDL",
    "layoffs_and_discharges_fiance_and_insurance" : "JTU5200LDL",
    "layoffs_and_discharges_construction" : "JTU2300LDL",
    "layoffs_and_discharges_total_private" : "JTS1000LDL",
    "layoffs_and_discharges_retail_trade" : "JTU4400LDL",
    "layoffs_and_discharges_real_estate_and_rental_and_leasing" :"JTU5300LDL",
    "layoffs_and_discharges_accommodation_and_food_services" : "JTU7200LDL",
    "real_estate_loans_all_commercial_bank" : "CREACBM027NBOG",
    "commercial_real_estate_prices_for_US_rate" : "COMREPUSQ159N"
}

spark_dataframes = fred_loader.fetch_multiple_indicators(macro_indicators, start="2010-01-01")

In [0]:
min_date = datetime(2010, 1, 1)
max_date = datetime(2010, 1, 1)

for key in spark_dataframes:
    min_date_, max_date_ = spark_dataframes[key].selectExpr("min(date)", "max(date)").first()
    if min_date_ < min_date:
        min_date = min_date_
    if max_date_ > max_date:
        max_date = max_date_

date_df = spark.sql(f"""
  SELECT sequence(to_date('{min_date}'), to_date('{max_date}'), interval 1 day) as all_dates
""").selectExpr("explode(all_dates) as date")

In [0]:
for key in spark_dataframes:
    df_filled = date_df.join(spark_dataframes[key], on="date", how="left")
    window_spec = Window.orderBy("date").rowsBetween(-sys.maxsize, 0)
    list_columns = spark_dataframes[key].columns
    for column in list_columns:
        if column != "date":
            df_filled = df_filled.withColumn(f"{column}_filled", last(column, ignorenulls=True).over(window_spec))
            df_filled = df_filled.drop(column).withColumnRenamed(f"{column}_filled", column)
    spark_dataframes[key] = df_filled

In [0]:
joined_df = spark_dataframes["gdp"]
for key in spark_dataframes:
    if key != "gdp":
        joined_df = joined_df.join(spark_dataframes[key], ['date'], "inner")

In [0]:
joined_df.printSchema()

In [0]:
display(joined_df)

In [0]:
joined_df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Silver.Macro_Historical") \
    .option("user", connection_properties["user"]) \
    .option("password", connection_properties["password"]) \
    .option("driver", connection_properties["driver"]) \
    .mode("overwrite") \
    .option("batchsize", 10000) \
    .option("numPartitions", 8) \
    .save()

print("Data successfully written to Azure SQL Database.")

In [0]:
# IF NOT EXISTS (
#     SELECT * FROM INFORMATION_SCHEMA.TABLES 
#     WHERE TABLE_NAME = 'Macro_Historical' AND TABLE_SCHEMA = 'Silver'
# )
# BEGIN
# CREATE TABLE Silver.Macro_Historical (
#     [date] DATETIME(50) NOT NULL,
#     [gdp_value] FLOAT NOT NULL,
#     [real_gdp_value] FLOAT NOT NULL,
#     [ferfed_funds_effective_rate_value] FLOAT NOT NULL,
#     [labor_force_participant_rate_value] FLOAT NOT NULL,
#     [cpi_value] FLOAT NOT NULL,
#     [unemployment_value] FLOAT NOT NULL,
#     [interest_rate_value] FLOAT NOT NULL,
#     [job_openning_non_farm_value] FLOAT NOT NULL,
#     [hires_total_non_farm_value] FLOAT NOT NULL,
#     [quit_total_non_farm_value] FLOAT NOT NULL,
#     [layoff_discharge_non_farm_value] FLOAT NOT NULL,
#     [layoffs_and_discharges_professional_and_business_services_value] FLOAT NOT NULL,
#     [layoffs_and_discharges_manufacturing_value] FLOAT NOT NULL,
#     [layoffs_and_discharges_fiance_and_insurance_value] FLOAT NOT NULL,
#     [layoffs_and_discharges_construction_value] FLOAT NOT NULL,
#     [layoffs_and_discharges_total_private_value] FLOAT NOT NULL,
#     [layoffs_and_discharges_retail_trade_value] FLOAT NOT NULL,
#     [layoffs_and_discharges_real_estate_and_rental_and_leasing_value] FLOAT NOT NULL,
#     [layoffs_and_discharges_accommodation_and_food_services_value] FLOAT NOT NULL,
#     [real_estate_loans_all_commercial_bank_value] FLOAT NOT NULL,
#     [commercial_real_estate_prices_for_US_rate_value] FLOAT NOT NULL,
# );
# END